In [1]:
!pip install -q kagglehub soundfile

import os
import re
import sys
import time
import random
import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torchvision.models import resnet18
from torch.utils.data import Dataset, DataLoader
import kagglehub

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("проверка датасета")
dataset_path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
LABEL_TO_IDX = {g: i for i, g in enumerate(GENRES)}
PATTERN = re.compile(r'(blues|classical|country|disco|hiphop|jazz|metal|pop|reggae|rock)[\._-]?(\d{5})', re.IGNORECASE)

disk_audio_map = {}
for root, _, files in os.walk(dataset_path):
    for f in files:
        if f.lower().endswith('.wav'):
            m = PATTERN.search(f)
            if m:
                g, num = m.group(1).lower(), m.group(2)
                if g == 'jazz' and num == '00054': #делитаем битый файл
                    continue
                disk_audio_map[(g, num)] = os.path.join(root, f)

print(f"исправных аудиофайлов {len(disk_audio_map)}")

if not os.path.exists("gtzan_sturm_filter_3folds_stratified"):
    print("скачиваем сплиты Штурма")
    os.system("git clone -q https://github.com/julianofoleiss/gtzan_sturm_filter_3folds_stratified.git")

class GTZAN_SturmSplit(Dataset):
    def __init__(self, split_file, audio_map, sample_rate=22050, n_mels=128, n_fft=2048, hop_length=512):
        self.items = []
        self.cache = {}
        self.target_len = sample_rate * 30

        with open(split_file, 'r') as f:
            for line in f:
                m = PATTERN.search(line)
                if m:
                    g, num = m.group(1).lower(), m.group(2)
                    if (g, num) in audio_map:
                        self.items.append((audio_map[(g, num)], LABEL_TO_IDX[g]))

        print(f"сопоставлено треков из {os.path.basename(split_file)}: {len(self.items)}")

        self.mel_spectrogram = T.MelSpectrogram(
            sample_rate=sample_rate, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
        )
        self.amplitude_to_db = T.AmplitudeToDB()

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        if idx not in self.cache:
            file_path, label = self.items[idx]
            try:
                waveform, sr = torchaudio.load(file_path)
            except Exception:
                data, sr = sf.read(file_path, dtype='float32')
                waveform = torch.from_numpy(data).unsqueeze(0) if data.ndim == 1 else torch.from_numpy(data.T)

            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)

            if waveform.shape[1] > self.target_len:
                waveform = waveform[:, :self.target_len]
            elif waveform.shape[1] < self.target_len:
                waveform = torch.nn.functional.pad(waveform, (0, self.target_len - waveform.shape[1]))

            mel = self.amplitude_to_db(self.mel_spectrogram(waveform))
            mel = (mel - mel.mean()) / (mel.std() + 1e-6)
            self.cache[idx] = (mel, label)

        mel, label = self.cache[idx]
        return mel.repeat(3, 1, 1), label


train_file = "gtzan_sturm_filter_3folds_stratified/f1_train.txt"
test_file = "gtzan_sturm_filter_3folds_stratified/f1_evaluate.txt"
train_set = GTZAN_SturmSplit(train_file, disk_audio_map)
test_set = GTZAN_SturmSplit(test_file, disk_audio_map)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print(f"\nУстройство: {device} ({gpu_name})")

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 30
print(f"\n начинаем обучение, сид {SEED}) на {num_epochs} эпох")

total_start = time.time()
for epoch in range(num_epochs):
    epoch_start = time.time()
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        _, pred = torch.max(out, 1)
        total_train += y.size(0)
        correct_train += (pred == y).sum().item()

    train_loss /= total_train
    train_acc = 100.0 * correct_train / total_train
    epoch_time = time.time() - epoch_start

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"эпох {epoch+1:02d}/{num_epochs} | время: {epoch_time:.1f} с | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.1f}%")

total_time = time.time() - total_start

model.eval()
correct = 0
total = 0
test_loss = 0.0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        test_loss += loss.item() * x.size(0)
        _, predicted = torch.max(out, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()

test_loss /= total
accuracy = 100.0 * correct / total

print(f"GPU:           {gpu_name}")
print(f"Общее время:   {total_time:.1f} сек ({total_time/60:.2f} мин)")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {accuracy:.2f}%")

проверка датасета
Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
исправных аудиофайлов 999
скачиваем сплиты Штурма
сопоставлено треков из f1_train.txt: 632
сопоставлено треков из f1_evaluate.txt: 315

Устройство: cuda (Tesla T4)

 начинаем обучение, сид 42) на 30 эпох
эпох 01/30 | время: 58.2 с | Train Loss: 1.7576 | Train Acc: 36.6%
эпох 05/30 | время: 7.3 с | Train Loss: 0.8328 | Train Acc: 69.5%
эпох 10/30 | время: 7.8 с | Train Loss: 0.4916 | Train Acc: 83.5%
эпох 15/30 | время: 7.4 с | Train Loss: 0.2346 | Train Acc: 91.8%
эпох 20/30 | время: 7.5 с | Train Loss: 0.1313 | Train Acc: 95.4%
эпох 25/30 | время: 7.5 с | Train Loss: 0.0432 | Train Acc: 98.7%
эпох 30/30 | время: 7.4 с | Train Loss: 0.0162 | Train Acc: 99.5%
GPU:           Tesla T4
Общее время:   274.2 сек (4.57 мин)
Test Loss:     2.2280
Test Accuracy: 57.46%
